In [ ]:
# IMPORTAÇÕES E CONFIGURAÇÃO DO AGENTE DQN
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

class QNetwork(nn.Module):
    """Rede Neural profunda para aproximar a função Q."""
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.995, buffer_size=10000):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        self.policy_net = QNetwork(state_dim, action_dim)
        self.target_net = QNetwork(state_dim, action_dim)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = deque(maxlen=buffer_size)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
        return torch.argmax(q_values).item()

    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def update(self, batch_size):
        if len(self.memory) < batch_size:
            return None

        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states_t = torch.FloatTensor(states)
        actions_t = torch.LongTensor(actions).unsqueeze(1)
        rewards_t = torch.FloatTensor(rewards)
        next_states_t = torch.FloatTensor(next_states)
        dones_t = torch.FloatTensor(dones)

        current_q = self.policy_net(states_t).gather(1, actions_t).squeeze(1)

        with torch.no_grad():
            max_next_q = self.target_net(next_states_t).max(1)[0]
            target_q = rewards_t + (1 - dones_t) * self.gamma * max_next_q

        loss = nn.MSELoss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        return loss.item()

    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

In [ ]:
# TREINAMENTO

STATE_DIM = 5
ACTION_DIM = 5

MAX_FILA = 30
MAX_TEMPO = 60

def normalizar_estado(state):
    fila_NS, fila_LO, tempo_V, fase_atual, tempo_desde_troca = state
    return np.array([
        fila_NS / MAX_FILA,
        fila_LO / MAX_FILA,
        tempo_V / MAX_TEMPO,
        float(fase_atual),
        min(1.0, tempo_desde_troca / 20.0)
    ], dtype=np.float32)

def simulated_step(state, action, step_count):
    fila_NS, fila_LO, tempo_V, fase_atual, tempo_desde_troca = state

    if action == 1 and fase_atual != 0: fase_atual = 0; tempo_desde_troca = 0
    elif action == 2 and fase_atual != 1: fase_atual = 1; tempo_desde_troca = 0
    elif action == 3: tempo_V = min(MAX_TEMPO, tempo_V + 5)
    elif action == 4: tempo_V = max(10, tempo_V - 5)
    else: tempo_desde_troca += 1

    # Modelagem do cenário
    if step_count <= 20:
        # Madrugada: Raramente chega carro
        chegada_NS = 1 if np.random.random() < 0.2 else 0
        chegada_LO = 1 if np.random.random() < 0.1 else 0

    elif 20 < step_count <= 50:
        # Pico da Manhã: Forte fluxo no sentido Norte-Sul
        chegada_NS = np.random.randint(1, 4) # Chega muito carro
        chegada_LO = np.random.randint(0, 2) # Fluxo calmo

    elif 50 < step_count <= 80:
        # Horário Comercial: Fluxo constante e moderado em ambos os lados
        chegada_NS = np.random.randint(0, 2)
        chegada_LO = np.random.randint(0, 2)

    else:
        # Pico da Tarde: Retorno para casa, forte fluxo em Leste-Oeste
        chegada_NS = np.random.randint(0, 2)
        chegada_LO = np.random.randint(1, 4) # Chega muito carro

    # Capacidade de escoamento
    vazo_NS = 3 if fase_atual == 0 else 0
    vazo_LO = 3 if fase_atual == 1 else 0

    fila_NS = max(0, min(MAX_FILA, fila_NS + chegada_NS - vazo_NS))
    fila_LO = max(0, min(MAX_FILA, fila_LO + chegada_LO - vazo_LO))

    # Cálculo da recompensa escalada
    reward = -(fila_NS + fila_LO) / (MAX_FILA * 2)

    # Penalidades
    if action in [1, 2] and tempo_desde_troca < 5:
        reward -= 0.02
    if fila_NS > 20 or fila_LO > 20:
        reward -= 0.05

    next_state = [fila_NS, fila_LO, tempo_V, fase_atual, tempo_desde_troca]
    return np.array(next_state), reward, False

def treinar_agente():
    agent = DQNAgent(state_dim=5, action_dim=5, lr=1e-4, epsilon_decay=0.992)

    episodes = 1000
    batch_size = 64
    target_update_freq = 20

    reward_history = []
    loss_history = []
    best_avg_reward = -float('inf')
    patience, patience_counter = 40, 0

    for episode in range(1, episodes + 1):
        raw_state = np.array([12, 7, 25, 0, 10])
        ep_reward = 0
        ep_losses = []

        for step in range(100):
            state_norm = normalizar_estado(raw_state)
            action = agent.select_action(state_norm)

            next_raw_state, reward, done = simulated_step(raw_state, action, step)
            next_state_norm = normalizar_estado(next_raw_state)

            agent.store_transition(state_norm, action, reward, next_state_norm, done)

            if len(agent.memory) >= batch_size:
                batch = random.sample(agent.memory, batch_size)
                states_b, actions_b, rewards_b, next_states_b, dones_b = zip(*batch)

                states_t = torch.FloatTensor(np.array(states_b))
                actions_t = torch.LongTensor(actions_b).unsqueeze(1)
                rewards_t = torch.FloatTensor(rewards_b)
                next_states_t = torch.FloatTensor(np.array(next_states_b))
                dones_t = torch.FloatTensor(dones_b)

                current_q = agent.policy_net(states_t).gather(1, actions_t).squeeze(1)
                with torch.no_grad():
                    max_next_q = agent.target_net(next_states_t).max(1)[0]
                    target_q = rewards_t + (1 - dones_t) * agent.gamma * max_next_q

                loss = nn.MSELoss()(current_q, target_q)
                agent.optimizer.zero_grad()
                loss.backward()
                agent.optimizer.step()
                ep_losses.append(loss.item())

            raw_state = next_raw_state
            ep_reward += reward

        if agent.epsilon > agent.epsilon_min:
            agent.epsilon *= agent.epsilon_decay

        if episode % target_update_freq == 0:
            agent.update_target_network()

        reward_history.append(ep_reward)
        loss_history.append(np.mean(ep_losses) if ep_losses else 0)

        running_reward = np.mean(reward_history[-10:])
        print(f"Episódio {episode:03d}/{episodes} | Recompensa: {ep_reward:.2f} | Média Móvel (10 ep): {running_reward:.2f} | Epsilon: {agent.epsilon:.2f}")

        if running_reward > best_avg_reward:
            best_avg_reward = running_reward
            patience_counter = 0
            torch.save(agent.policy_net.state_dict(), "best_traffic_dqn.pt")
            print("=> [Sinal Verde] Modelo Excelente Guardado!")
        else:
            if episode > 60:
                patience_counter += 1

        if patience_counter >= patience:
            print(f"\n[Early Stopping] Modelo Convergido no episódio {episode}.")
            break

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(reward_history, alpha=0.3, color="b", label="Recompensa Real")
    sma = np.convolve(reward_history, np.ones(10)/10, mode='valid')
    plt.plot(range(9, len(reward_history)), sma, color="darkblue", linewidth=2, label="Tendência (Média Móvel)")
    plt.title("Curva de Aprendizado (Alvo Próximo a Zero)")
    plt.xlabel("Episódios"); plt.ylabel("Recompensa Acumulada"); plt.grid(True); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss_history, color="purple")
    plt.title("Evolução do Erro da Rede (Loss)")
    plt.xlabel("Episódios"); plt.ylabel("Valor do Erro"); plt.grid(True)

    plt.tight_layout()
    plt.savefig("grafico_convergencia.png", dpi=300)
    plt.show()

import random
treinar_agente()

In [ ]:
from google.colab import files
files.download('best_traffic_dqn.pt')